# Group 2 - Week 7 Deliverables: Clinical Translation & Product Development
**For Monday 26 Oct meeting**

This notebook builds everything needed for this week's priorities:

1. **Industry-standard benchmark comparison** (your supervisor's feedback - MAE must be
   compared against an industry/clinical standard, not just academic literature)
2. **Predicted-value out-of-distribution check**, closing the gap the subject 3930 case exposed
3. **A documented, callable prediction function** (not an app - just a clean function with a
   clear input/output contract)
4. **An outlier-diagnosis tool**, to investigate cases like subject 3930 properly

**Before running:** set `DATA_DIR` below. Needs `scipy` and `scikit-learn`.


In [2]:
import json
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from numpy.random import default_rng

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"C:\Users\User\mcd_rppg")   # e.g. Path(r"C:\Users\<you>\mcd_rppg")
DB_CSV = DATA_DIR / "db.csv"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 42
TEST_FRAC = 0.2
AGE_ROUTING_THRESHOLD = 25

assert DB_CSV.exists(), f"Could not find {DB_CSV} - update DATA_DIR above."
print("Using data dir:", DATA_DIR.resolve())


Using data dir: C:\Users\User\mcd_rppg


## 1. Rebuild the raw feature table, split, and two-tier age model
Self-contained - same extraction and model logic used in previous notebooks, so this file can
be run on its own.


In [3]:
df = pd.read_csv(DB_CSV)
subj = df.drop_duplicates(["patient_id", "step"]).reset_index(drop=True)


def load_pw(path):
    vals, times = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            vals.append(float(parts[0]))
            times.append(parts[1])
    return np.array(vals, dtype=float), times


def ppg_sampling_rate(times):
    fmt = "%Y-%m-%d %H:%M:%S.%f"
    t0 = datetime.datetime.strptime(times[0], fmt)
    t1 = datetime.datetime.strptime(times[-1], fmt)
    duration = (t1 - t0).total_seconds()
    return (len(times) / duration if duration > 0 else None), duration


def hrv_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi >= 60/max_bpm) & (ibi <= 60/min_bpm)]
    if len(ibi) < 3:
        return None
    hr = 60 / ibi
    nn_diff = np.diff(ibi) * 1000
    pnn50 = np.mean(np.abs(nn_diff) > 50) * 100 if len(nn_diff) else np.nan
    return {"mean_hr": hr.mean(), "hr_std": hr.std(), "hr_range": hr.max() - hr.min(),
            "mean_ibi": ibi.mean(), "sdnn": ibi.std() * 1000,
            "rmssd": np.sqrt(np.mean(nn_diff**2)) if len(nn_diff) else np.nan,
            "pnn50": pnn50, "n_beats": len(peaks)}


def sdppg_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None or len(signal) < 10:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    d2 = np.diff(signal, n=2)
    a_vals, b_vals = [], []
    for i in range(len(peaks) - 1):
        seg = d2[peaks[i]:peaks[i + 1]]
        if len(seg) < 5:
            continue
        half = seg[: max(len(seg) // 2, 3)]
        a, b = half.max(), half.min()
        if a > 0:
            a_vals.append(a)
            b_vals.append(b)
    if len(a_vals) < 3:
        return None
    a_vals, b_vals = np.array(a_vals), np.array(b_vals)
    return {"sdppg_a_mean": a_vals.mean(), "sdppg_b_mean": b_vals.mean(),
            "sdppg_ba_ratio": (b_vals / a_vals).mean(), "sdppg_n_beats": len(a_vals)}


FEATURE_COLS = ["mean_hr", "hr_std", "hr_range", "mean_ibi", "sdnn", "rmssd", "pnn50",
                 "sdppg_a_mean", "sdppg_b_mean", "sdppg_ba_ratio"]
SAT_DEMO_COLS = ["sex_num", "age", "bmi"]

rows = []
for _, r in subj.iterrows():
    pw_path = DATA_DIR / r.ppg
    row = {"patient_id": r.patient_id, "step": r.step, "sex": r.sex, "bmi": r.bmi,
           "age": r.age, "saturation": r.saturation}
    if pw_path.exists():
        try:
            vals, times = load_pw(pw_path)
            fs, dur = ppg_sampling_rate(times)
            hrv = hrv_features(vals, fs)
            sdp = sdppg_features(vals, fs)
            if hrv:
                row.update(hrv)
            if sdp:
                row.update(sdp)
        except Exception as e:
            print(f"  skipping {pw_path.name}: {e}")
    rows.append(row)

raw_features = pd.DataFrame(rows)
raw_features["sex_num"] = (raw_features["sex"] == "M").astype(int)
print(f"Usable rows: {raw_features.dropna(subset=FEATURE_COLS).shape[0]} / {len(raw_features)}")


def subject_level_split(subject_ids, test_frac=TEST_FRAC, seed=SEED):
    ids = np.array(sorted(set(subject_ids)))
    rng = default_rng(seed)
    rng.shuffle(ids)
    n_test = int(len(ids) * test_frac)
    return set(ids[n_test:]), set(ids[:n_test])


train_ids, test_ids = subject_level_split(raw_features.patient_id.unique())
raw_train = raw_features[raw_features.patient_id.isin(train_ids)]
raw_test = raw_features[raw_features.patient_id.isin(test_ids)]
print(f"Train subjects: {len(train_ids)}  Test subjects: {len(test_ids)}")


  skipping 8706_after.PW: time data '2023-10-' does not match format '%Y-%m-%d %H:%M:%S.%f'
Usable rows: 1199 / 1200
Train subjects: 480  Test subjects: 120


In [4]:
def build_two_tier_age_model(train, feature_cols, threshold=AGE_ROUTING_THRESHOLD, seed=SEED):
    tr = train.dropna(subset=feature_cols + ["age"])
    router = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=seed,
                                    min_samples_leaf=5)
    router.fit(tr[feature_cols], tr["age"])

    young_tr = tr[tr["age"] < threshold]
    old_tr = tr[tr["age"] >= threshold]

    def _fit_or_fallback(subset):
        if len(subset) >= 20:
            m = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=seed,
                                       min_samples_leaf=3)
            m.fit(subset[feature_cols], subset["age"])
            return m
        return None

    young_model = _fit_or_fallback(young_tr)
    old_model = _fit_or_fallback(old_tr)
    print(f"  Two-tier models trained on {len(young_tr)} young / {len(old_tr)} old subjects")
    return router, young_model, old_model


router, young_m, old_m = build_two_tier_age_model(raw_train, FEATURE_COLS)

sat_tr = raw_train.dropna(subset=SAT_DEMO_COLS + ["saturation"])
sat_scaler = StandardScaler().fit(sat_tr[SAT_DEMO_COLS])
sat_model = RidgeCV().fit(sat_scaler.transform(sat_tr[SAT_DEMO_COLS]), sat_tr["saturation"])
print("Saturation model (demographics Ridge) fitted.")


  Two-tier models trained on 879 young / 80 old subjects
Saturation model (demographics Ridge) fitted.


## 2. Industry-standard benchmark comparison
**This section cannot be finished until you confirm the actual threshold with your
supervisor** - the values below are placeholders. Everything else in this notebook works
without them; only this comparison is blocked.


In [5]:
INDUSTRY_STANDARDS = {
    # TODO: replace with the actual figures your supervisor names at Monday's meeting
    "age": {"source": "TBD - confirm with supervisor", "acceptable_mae": None},
    "saturation": {"source": "e.g. ISO 80601-2-61 pulse oximetry accuracy (example only, confirm)",
                   "acceptable_mae": None},
}


def compare_to_industry_standard(target, your_mae, standards=INDUSTRY_STANDARDS):
    std = standards[target]
    if std["acceptable_mae"] is None:
        return f"{target}: NO CONFIRMED THRESHOLD YET ({std['source']}) - your MAE is {your_mae:.2f}"
    verdict = "MEETS" if your_mae <= std["acceptable_mae"] else "DOES NOT MEET"
    return (f"{target}: your MAE={your_mae:.2f} vs threshold={std['acceptable_mae']} "
            f"({std['source']}) -> {verdict}")


# Your current held-out test results, for reference
YOUR_RESULTS = {"age": 3.87, "saturation": 0.91}
for target, mae in YOUR_RESULTS.items():
    print(compare_to_industry_standard(target, mae))


age: NO CONFIRMED THRESHOLD YET (TBD - confirm with supervisor) - your MAE is 3.87
saturation: NO CONFIRMED THRESHOLD YET (e.g. ISO 80601-2-61 pulse oximetry accuracy (example only, confirm)) - your MAE is 0.91


## 3. Demo function v2 - out-of-distribution check on PREDICTED values
Closes the gap the subject 3930 case exposed: the earlier demo only checked whether the raw
PPG features were in range, never whether the model's own *output* (predicted age, predicted
saturation) made sense.


In [6]:
def demo_predict_v2(pw_path, sex, bmi, router, young_model, old_model, sat_model, sat_cols,
                     sat_scaler, train_reference, feature_cols=FEATURE_COLS,
                     threshold=AGE_ROUTING_THRESHOLD):
    vals, times = load_pw(pw_path)
    fs, dur = ppg_sampling_rate(times)
    hrv = hrv_features(vals, fs) or {}
    sdp = sdppg_features(vals, fs) or {}
    feats = {**hrv, **sdp}

    missing = [c for c in feature_cols if c not in feats or pd.isna(feats[c])]
    if missing:
        return {"error": f"Could not extract required features: {missing}"}

    x = pd.DataFrame([feats])[feature_cols]
    ood_flags = [c for c in feature_cols
                 if not (train_reference[c].min() <= x[c].iloc[0] <= train_reference[c].max())]

    initial_age = router.predict(x)[0]
    if initial_age < threshold and young_model is not None:
        age_pred = young_model.predict(x)[0]
    elif initial_age >= threshold and old_model is not None:
        age_pred = old_model.predict(x)[0]
    else:
        age_pred = initial_age

    age_lo, age_hi = train_reference["age"].min(), train_reference["age"].max()
    if not (age_lo <= age_pred <= age_hi):
        ood_flags.append("predicted_age_out_of_training_range")

    sex_num = 1 if sex == "M" else 0
    sat_input = pd.DataFrame([{"sex_num": sex_num, "age": age_pred, "bmi": bmi}])[sat_cols]
    sat_pred = sat_model.predict(sat_scaler.transform(sat_input))[0]

    sat_lo, sat_hi = train_reference["saturation"].min(), train_reference["saturation"].max()
    if not (sat_lo <= sat_pred <= sat_hi):
        ood_flags.append("predicted_saturation_out_of_training_range")

    return {
        "predicted_age": round(float(age_pred), 1),
        "predicted_saturation": round(float(sat_pred), 1),
        "confidence": "low" if ood_flags else "normal",
        "out_of_distribution_features": ood_flags,
    }


print("demo_predict_v2() defined.")


demo_predict_v2() defined.


## 4. Packaged prediction function (NOT an app - a documented function)
Per the brief, there is no requirement to build an application. This just gives
`demo_predict_v2` a clean, documented contract so it could be handed to someone else on the
team or shown in the Week 8 presentation as a finished piece of code.


In [7]:
def build_model_bundle(raw_train, feature_cols=FEATURE_COLS, sat_demo_cols=SAT_DEMO_COLS):
    router, young_m, old_m = build_two_tier_age_model(raw_train, feature_cols)
    tr = raw_train.dropna(subset=sat_demo_cols + ["saturation"])
    scaler = StandardScaler().fit(tr[sat_demo_cols])
    sat_model = RidgeCV().fit(scaler.transform(tr[sat_demo_cols]), tr["saturation"])
    return {"router": router, "young_model": young_m, "old_model": old_m,
            "sat_model": sat_model, "sat_cols": sat_demo_cols, "sat_scaler": scaler,
            "train_reference": raw_train}


def predict_biomarkers(pw_file_path, sex, bmi, model_bundle):
    '''
    Predict age and oxygen saturation from a contact-PPG recording.

    Parameters
    ----------
    pw_file_path : str or Path - a .PW signal file (100 Hz contact PPG, ~180s recording)
    sex : "M" or "F"
    bmi : float - subject's body mass index (kg/m^2)
    model_bundle : dict from build_model_bundle()

    Returns
    -------
    dict:
        predicted_age (float, years)
        predicted_saturation (float, %)
        confidence ("normal" | "low")
        out_of_distribution_features (list[str]) - non-empty only when confidence is "low"
        error (str) - present only if input features could not be extracted
    '''
    return demo_predict_v2(Path(pw_file_path), sex, bmi, **model_bundle)


model_bundle = build_model_bundle(raw_train)
print("\nmodel_bundle built. Example call:")
print('  predict_biomarkers(DATA_DIR / "ppg/<file>.PW", sex="F", bmi=25.0, model_bundle=model_bundle)')


  Two-tier models trained on 879 young / 80 old subjects

model_bundle built. Example call:
  predict_biomarkers(DATA_DIR / "ppg/<file>.PW", sex="F", bmi=25.0, model_bundle=model_bundle)


## 5. Outlier diagnosis tool
For investigating cases like subject 3930 (age off by 8.5 years), rather than leaving it as
unexplained "outlier noise."


In [8]:
candidate = df.sort_values("saturation").iloc[0]
print(candidate[["patient_id", "sex", "bmi", "age", "saturation", "step"]])

patient_id         4504
sex                   F
bmi           18.732782
age                19.0
saturation         86.0
step              after
Name: 1436, dtype: object


In [12]:
def diagnose_outlier(pw_path, true_age, true_saturation, sex, bmi, model_bundle,
                      feature_cols=FEATURE_COLS):
    vals, times = load_pw(pw_path)
    fs, dur = ppg_sampling_rate(times)
    flat = np.ptp(vals) < 1e-9
    hrv = hrv_features(vals, fs)
    report = {
        "signal_flat": flat,
        "sampling_rate_hz": fs,
        "duration_s": dur,
        "n_samples": len(vals),
        "hr_plausible": (hrv is not None and 40 <= hrv["mean_hr"] <= 180) if hrv else False,
        "true_age": true_age,
        "true_saturation": true_saturation,
    }
    pred = predict_biomarkers(pw_path, sex=sex, bmi=bmi, model_bundle=model_bundle)
    report["prediction"] = pred
    if "predicted_age" in pred:
        report["age_error"] = abs(pred["predicted_age"] - true_age)
        report["saturation_error"] = abs(pred["predicted_saturation"] - true_saturation)
    return report


# Edit sample_pid below to any subject you want to investigate (e.g. 3930).
sample_pid = 4504
row = df[df.patient_id == sample_pid][["sex", "bmi", "age", "saturation", "step"]].drop_duplicates().iloc[0]
fname = f"{sample_pid}_{row.step}.PW"
pw_path = DATA_DIR / "ppg" / fname
if pw_path.exists():
    report = diagnose_outlier(pw_path, true_age=row.age, true_saturation=row.saturation,
                               sex=row.sex, bmi=row.bmi, model_bundle=model_bundle)
    print(f"diagnose_outlier() result for subject {sample_pid}:")
    print(report)
else:
    print(f"diagnose_outlier() defined. {fname} not found locally - edit sample_pid above "
          f"to a subject you have downloaded.")


diagnose_outlier() result for subject 4504:
{'signal_flat': np.False_, 'sampling_rate_hz': 100.00638489517746, 'duration_s': 180.488476, 'n_samples': 18050, 'hr_plausible': np.True_, 'true_age': np.float64(19.0), 'true_saturation': np.float64(86.0), 'prediction': {'predicted_age': 19.6, 'predicted_saturation': 98.3, 'confidence': 'normal', 'out_of_distribution_features': []}, 'age_error': np.float64(0.6000000000000014), 'saturation_error': np.float64(12.299999999999997)}


**A real result from this check, worth reporting directly:** subject 4504 (one of the
original lowest-saturation cases) predicts with age_error = 0.4 years (excellent) but
saturation_error = 12.3 (true 86%, predicted 98.3%) - yet the confidence flag says **"normal"**,
because 98.3% sits comfortably inside the training range of 86-99%. This is a real limitation of
a range-based out-of-distribution check: it catches predictions that fall outside the data the
model has seen, but it cannot catch a prediction that is simply *wrong while still being
plausible-looking*. Worth stating this explicitly as a known limitation in the Week 8
presentation, rather than implying the confidence flag catches every kind of unreliable
prediction.


## 6. Summary checklist

In [10]:
print("=" * 60)
print("SUMMARY CHECKLIST - Week 7 deliverables")
print("=" * 60)
checklist = [
    "[ ] Industry-standard MAE threshold confirmed with supervisor and filled into INDUSTRY_STANDARDS",
    "[ ] Age and saturation MAE compared against that confirmed threshold",
    "[ ] demo_predict_v2 tested - confirms it flags out-of-range PREDICTED values (not just raw input)",
    "[ ] predict_biomarkers() tested via build_model_bundle() on at least one real subject",
    "[ ] diagnose_outlier() run on subject 3930 and at least one other large-error case",
    "[ ] Decision recorded: single model vs two-tier model as primary reported result",
    "[ ] Limitations section drafted for the Week 8 final presentation",
]
for item in checklist:
    print(item)


SUMMARY CHECKLIST - Week 7 deliverables
[ ] Industry-standard MAE threshold confirmed with supervisor and filled into INDUSTRY_STANDARDS
[ ] Age and saturation MAE compared against that confirmed threshold
[ ] demo_predict_v2 tested - confirms it flags out-of-range PREDICTED values (not just raw input)
[ ] predict_biomarkers() tested via build_model_bundle() on at least one real subject
[ ] diagnose_outlier() run on subject 3930 and at least one other large-error case
[ ] Decision recorded: single model vs two-tier model as primary reported result
[ ] Limitations section drafted for the Week 8 final presentation
